# F1 Pit Stops

EDA, feature engineering, and a 4-model ensemble (LightGBM, bagged CatBoost,
XGBoost, sklearn MLP) for [Playground Series S6E5](https://www.kaggle.com/competitions/playground-series-s6e5).
Best own-model OOF AUC ~0.949; the final submission is produced by `blend.py`.

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.collections import LineCollection
from scipy.special import expit, logit
from scipy.stats import rankdata
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler

import lightgbm as lgb
from catboost import CatBoostClassifier
import xgboost as xgb

from features import CAT_COLS, TARGET, build_features

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid', context='notebook')

SEED = 42
N_FOLDS = 5
DATA = Path('data')
OOF = Path('oof'); OOF.mkdir(exist_ok=True)

## Load

In [ ]:
train = pd.read_csv(DATA / 'train.csv')
test  = pd.read_csv(DATA / 'test.csv')
train.rename(columns={'LapTime (s)': 'LapTime'}, inplace=True)
test.rename(columns={'LapTime (s)': 'LapTime'}, inplace=True)
print(train.shape, test.shape, 'pos rate', round(train[TARGET].mean(), 4))

## EDA

In [ ]:
by_comp = (train.groupby('Compound')[TARGET]
                .agg(['mean', 'count'])
                .sort_values('count', ascending=False))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
by_comp['mean'].plot.bar(ax=axes[0], color='steelblue', edgecolor='k')
axes[0].set(title='P(pit next lap) by compound', ylabel='P(pit)', xlabel='')
by_comp['count'].plot.bar(ax=axes[1], color='gray', edgecolor='k')
axes[1].set(title='Sample count by compound', ylabel='rows', xlabel='')
plt.tight_layout(); plt.show()
by_comp

In [ ]:
# 2023 is an anomaly — pit rate ~1% vs ~28% elsewhere. Track per-year AUC later.
yr = train.groupby('Year').agg(rows=('id', 'count'), pit_rate=(TARGET, 'mean'))
print(yr)
fig, ax = plt.subplots(figsize=(7, 3.5))
yr['pit_rate'].plot.bar(ax=ax, color='steelblue', edgecolor='k')
ax.set(title='Pit rate by year', ylabel='P(pit)')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for comp in ['SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE', 'WET']:
    sub = train[train['Compound'] == comp]
    if len(sub) < 100:
        continue
    bins = np.arange(0, sub['TyreLife'].max() + 2, 2)
    rate = sub.assign(b=pd.cut(sub['TyreLife'], bins)).groupby('b')[TARGET].mean()
    ax.plot([b.mid for b in rate.index], rate.values, marker='o', label=comp, lw=2)
ax.set(xlabel='TyreLife (laps)', ylabel='P(pit next lap)', title='Pit hazard by compound')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
h = train.assign(tl_bin=pd.cut(train['TyreLife'], bins=np.arange(0, 50, 3)))
pivot = h.groupby(['Compound', 'tl_bin'])[TARGET].mean().unstack()
fig, ax = plt.subplots(figsize=(13, 3.5))
sns.heatmap(pivot, cmap='RdYlBu_r', cbar_kws={'label': 'P(pit)'}, ax=ax)
ax.set(title='P(pit next lap) — Compound × TyreLife', xlabel='TyreLife bin', ylabel='')
plt.tight_layout(); plt.show()

In [ ]:
# Stylised circuit avatars coloured by pit risk along RaceProgress.
def track_xy(seed, n=400):
    rng = np.random.default_rng(seed)
    theta = np.linspace(0, 2 * np.pi, n, endpoint=False)
    r = np.ones_like(theta)
    for k in range(2, 8):
        r = r + rng.uniform(-0.18, 0.18) / (k * 0.6) * np.cos(k * theta + rng.uniform(0, 2 * np.pi))
    return r * np.cos(theta), r * np.sin(theta), theta

n_bins = 60
tmp = train.assign(rp_bin=(train['RaceProgress'].clip(0, 1 - 1e-6) * n_bins).astype(int))
race_rate = tmp.groupby(['Race', 'rp_bin'])[TARGET].mean().unstack(fill_value=np.nan)
races = (race_rate.max(1) - race_rate.min(1)).sort_values(ascending=False).head(9).index.tolist()

fig, axes = plt.subplots(3, 3, figsize=(13, 13))
vmax = float(np.nanpercentile(race_rate.values, 99))
for ax, race in zip(axes.flat, races):
    x, y_, theta = track_xy(seed=abs(hash(race)) % (2**32))
    bin_idx = ((theta / (2 * np.pi)) * n_bins).astype(int)
    rate_per_pt = race_rate.loc[race].reindex(range(n_bins)).values[bin_idx]
    pts = np.array([x, y_]).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc = LineCollection(segs, cmap='RdYlBu_r', linewidth=10, norm=plt.Normalize(0, vmax))
    lc.set_array(rate_per_pt[:-1])
    ax.add_collection(lc)
    ax.plot(x[0], y_[0], 'ks', markersize=7)
    ax.text(x[0] + 0.05, y_[0] + 0.05, 'S/F', fontsize=8)
    ax.set_xlim(-1.35, 1.35); ax.set_ylim(-1.35, 1.35); ax.set_aspect('equal'); ax.axis('off')
    n_rows = int((train['Race'] == race).sum())
    pit_rate = train.loc[train['Race'] == race, TARGET].mean()
    ax.set_title(f'{race}\nrows={n_rows:,}  pit rate={pit_rate:.3f}', fontsize=10)
fig.suptitle('Pit risk along stylised track avatars', fontsize=14, y=1.0)
fig.colorbar(lc, ax=axes, shrink=0.5, label='P(pit next lap)')
plt.show()

In [ ]:
stint_lens = (train.groupby(['Race', 'Year', 'Driver', 'Stint', 'Compound'])
                   .size()
                   .reset_index(name='laps'))
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=stint_lens, x='Compound', y='laps',
            order=['SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE', 'WET'], ax=ax)
ax.set(title='Stint length by compound', ylabel='laps in stint')
plt.tight_layout(); plt.show()

## Features

In [ ]:
X, X_te, y, feature_cols, folds, test_ids_sorted = build_features(
    data_dir=str(DATA), n_folds=N_FOLDS, seed=SEED,
)
print(f'features: {len(feature_cols)}  train: {X.shape}  test: {X_te.shape}')

## Cached training

In [ ]:
def cached(name, train_fn):
    path = OOF / f'{name}.npz'
    if path.exists():
        data = np.load(path)
        oof, pred = data['oof'], data['pred']
        print(f'{name} cached  OOF AUC {roc_auc_score(y, oof):.5f}')
        return oof, pred
    oof, pred = train_fn()
    np.savez(path, oof=oof, pred=pred)
    return oof, pred

## LightGBM

In [ ]:
lgb_params = dict(
    objective='binary', metric='auc',
    learning_rate=0.03, num_leaves=511, min_child_samples=30,
    feature_fraction=0.85, bagging_fraction=0.85, bagging_freq=1,
    lambda_l2=2.0, verbose=-1, seed=SEED, n_jobs=-1,
)
lgb_models = []

def train_lgb():
    oof = np.zeros(len(X))
    pred = np.zeros(len(X_te))
    for f, (tr, va) in enumerate(folds):
        t0 = time.time()
        dtr = lgb.Dataset(X.iloc[tr], y[tr], categorical_feature=CAT_COLS)
        dva = lgb.Dataset(X.iloc[va], y[va], categorical_feature=CAT_COLS)
        m = lgb.train(
            lgb_params, dtr, 6000, valid_sets=[dva],
            callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)],
        )
        oof[va] = m.predict(X.iloc[va], num_iteration=m.best_iteration)
        pred += m.predict(X_te, num_iteration=m.best_iteration) / N_FOLDS
        lgb_models.append(m)
        print(f'  fold {f+1}: AUC {roc_auc_score(y[va], oof[va]):.5f}  '
              f'iter {m.best_iteration}  {time.time()-t0:.0f}s')
    return oof, pred

oof_lgb, pred_lgb = cached('lgb', train_lgb)
auc_lgb = roc_auc_score(y, oof_lgb)

## CatBoost (3-seed bag)

In [ ]:
X_cb, X_te_cb = X.copy(), X_te.copy()
for c in CAT_COLS:
    X_cb[c]    = X_cb[c].astype(str).fillna('NA')
    X_te_cb[c] = X_te_cb[c].astype(str).fillna('NA')

CB_CONFIGS = [
    dict(seed=42,   depth=8, learning_rate=0.05, l2_leaf_reg=3.0, bagging_temperature=0.8, random_strength=1.0),
    dict(seed=7,    depth=7, learning_rate=0.06, l2_leaf_reg=2.0, bagging_temperature=1.0, random_strength=1.5),
    dict(seed=2023, depth=9, learning_rate=0.04, l2_leaf_reg=4.0, bagging_temperature=0.6, random_strength=0.8),
]

def train_cb(cfg):
    oof = np.zeros(len(X))
    pred = np.zeros(len(X_te))
    for f, (tr, va) in enumerate(folds):
        t0 = time.time()
        m = CatBoostClassifier(
            iterations=4000, **{k: v for k, v in cfg.items() if k != 'seed'},
            random_seed=cfg['seed'], eval_metric='AUC',
            cat_features=CAT_COLS, early_stopping_rounds=120,
            verbose=0, task_type='CPU',
        )
        m.fit(X_cb.iloc[tr], y[tr], eval_set=(X_cb.iloc[va], y[va]), use_best_model=True)
        oof[va] = m.predict_proba(X_cb.iloc[va])[:, 1]
        pred += m.predict_proba(X_te_cb)[:, 1] / N_FOLDS
        print(f'    fold {f+1}: AUC {roc_auc_score(y[va], oof[va]):.5f}  '
              f'iter {m.tree_count_}  {time.time()-t0:.0f}s')
    return oof, pred

oof_cb_all, pred_cb_all = [], []
for i, cfg in enumerate(CB_CONFIGS):
    print(f'\nCB[{i+1}/{len(CB_CONFIGS)}] {cfg}')
    o, p = cached(f'cb_{i}', lambda cfg=cfg: train_cb(cfg))
    oof_cb_all.append(o); pred_cb_all.append(p)

def to_rank(x, eps=1e-6):
    return np.clip(rankdata(x) / len(x), eps, 1 - eps)

def logit_rank(preds, weights):
    z = sum(w * logit(to_rank(p)) for w, p in zip(weights, preds))
    return expit(z / sum(weights))

aucs_cb = [roc_auc_score(y, o) for o in oof_cb_all]
w_cb = np.array(aucs_cb) ** 2
oof_cb  = logit_rank(oof_cb_all,  w_cb)
pred_cb = logit_rank(pred_cb_all, w_cb)
auc_cb  = roc_auc_score(y, oof_cb)
print('\nCB per-seed OOF:', [round(a, 5) for a in aucs_cb])
print(f'CB bagged OOF AUC: {auc_cb:.5f}')

## XGBoost

In [ ]:
X_xgb, X_te_xgb = X.copy(), X_te.copy()
for c in CAT_COLS:
    X_xgb[c] = X_xgb[c].astype('category')
    X_te_xgb[c] = X_te_xgb[c].astype('category')

xgb_params = dict(
    objective='binary:logistic', eval_metric='auc',
    tree_method='hist', enable_categorical=True,
    learning_rate=0.05, max_depth=8,
    min_child_weight=20, subsample=0.85, colsample_bytree=0.85,
    reg_lambda=2.0, seed=SEED, n_jobs=-1, verbosity=0,
)

def train_xgb():
    oof = np.zeros(len(X))
    pred = np.zeros(len(X_te))
    dte = xgb.DMatrix(X_te_xgb, enable_categorical=True)
    for f, (tr, va) in enumerate(folds):
        t0 = time.time()
        dtr = xgb.DMatrix(X_xgb.iloc[tr], label=y[tr], enable_categorical=True)
        dva = xgb.DMatrix(X_xgb.iloc[va], label=y[va], enable_categorical=True)
        m = xgb.train(
            xgb_params, dtr, num_boost_round=4000,
            evals=[(dva, 'va')], early_stopping_rounds=120, verbose_eval=0,
        )
        oof[va] = m.predict(dva, iteration_range=(0, m.best_iteration + 1))
        pred += m.predict(dte, iteration_range=(0, m.best_iteration + 1)) / N_FOLDS
        print(f'  fold {f+1}: AUC {roc_auc_score(y[va], oof[va]):.5f}  '
              f'iter {m.best_iteration}  {time.time()-t0:.0f}s')
    return oof, pred

oof_xgb, pred_xgb = cached('xgb', train_xgb)
auc_xgb = roc_auc_score(y, oof_xgb)

## sklearn MLP

In [ ]:
X_mlp    = X.copy()
X_te_mlp = X_te.copy()
for c in CAT_COLS:
    le = LabelEncoder().fit(pd.concat([X_mlp[c].astype(str), X_te_mlp[c].astype(str)]))
    X_mlp[c]    = le.transform(X_mlp[c].astype(str)).astype(np.float32)
    X_te_mlp[c] = le.transform(X_te_mlp[c].astype(str)).astype(np.float32)
for c in feature_cols:
    if X_mlp[c].isnull().any() or X_te_mlp[c].isnull().any():
        med = X_mlp[c].median()
        X_mlp[c]    = X_mlp[c].fillna(med)
        X_te_mlp[c] = X_te_mlp[c].fillna(med)
scaler = StandardScaler()
X_mlp    = np.clip(scaler.fit_transform(X_mlp.astype(np.float64)), -10, 10)
X_te_mlp = np.clip(scaler.transform(X_te_mlp.astype(np.float64)), -10, 10)

def train_mlp():
    oof = np.zeros(len(X))
    pred = np.zeros(len(X_te))
    for f, (tr, va) in enumerate(folds):
        t0 = time.time()
        m = MLPClassifier(
            hidden_layer_sizes=(256, 128, 64),
            activation='relu', solver='adam',
            alpha=1e-4, batch_size=2048,
            learning_rate_init=1e-3, max_iter=40,
            early_stopping=True, validation_fraction=0.1,
            n_iter_no_change=5, random_state=SEED + f, verbose=False,
        )
        m.fit(X_mlp[tr], y[tr])
        oof[va] = m.predict_proba(X_mlp[va])[:, 1]
        pred += m.predict_proba(X_te_mlp)[:, 1] / N_FOLDS
        print(f'  fold {f+1}: AUC {roc_auc_score(y[va], oof[va]):.5f}  '
              f'iter {m.n_iter_}  {time.time()-t0:.0f}s')
    return oof, pred

oof_mlp, pred_mlp = cached('mlp', train_mlp)
auc_mlp = roc_auc_score(y, oof_mlp)

## Diagnostics

In [ ]:
oof_df = pd.DataFrame({
    'lgb': oof_lgb, 'cb_bag': oof_cb,
    'cb_0': oof_cb_all[0], 'cb_1': oof_cb_all[1], 'cb_2': oof_cb_all[2],
    'xgb': oof_xgb, 'mlp': oof_mlp,
})
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(oof_df.corr('pearson'),  annot=True, fmt='.4f', cmap='viridis', cbar=False, ax=axes[0])
axes[0].set_title('Pearson')
sns.heatmap(oof_df.corr('spearman'), annot=True, fmt='.4f', cmap='viridis', cbar=False, ax=axes[1])
axes[1].set_title('Spearman')
plt.tight_layout(); plt.show()

In [ ]:
tr_full = (pd.read_csv(DATA / 'train.csv')
             .rename(columns={'LapTime (s)': 'LapTime'})
             .sort_values(['Race', 'Year', 'Driver', 'LapNumber'])
             .reset_index(drop=True))

rows = []
for yr in sorted(tr_full['Year'].unique()):
    mask = tr_full['Year'].values == yr
    if y[mask].sum() in (0, mask.sum()):
        continue
    rows.append({
        'year': yr,
        'rows': int(mask.sum()),
        'pit_rate': float(y[mask].mean()),
        'cb_auc':  roc_auc_score(y[mask], oof_cb[mask]),
        'lgb_auc': roc_auc_score(y[mask], oof_lgb[mask]),
        'xgb_auc': roc_auc_score(y[mask], oof_xgb[mask]),
        'mlp_auc': roc_auc_score(y[mask], oof_mlp[mask]),
        'cb_pred_mean': float(oof_cb[mask].mean()),
    })
pd.DataFrame(rows).set_index('year').round(4)

In [ ]:
race_auc = (tr_full.assign(oof=oof_cb)
                   .groupby('Race')
                   .apply(lambda d: roc_auc_score(d[TARGET], d['oof'])
                          if d[TARGET].nunique() > 1 else np.nan)
                   .sort_values())
fig, ax = plt.subplots(figsize=(8, 8))
race_auc.head(20).plot.barh(ax=ax, color='steelblue')
ax.axvline(auc_cb, color='red', ls='--', label=f'overall {auc_cb:.4f}')
ax.set(title='20 weakest races (bagged CB OOF)', xlabel='AUC')
ax.legend(); plt.tight_layout(); plt.show()

## Stacker + isotonic

In [ ]:
models = {'lgb': (oof_lgb, pred_lgb), 'cb_bag': (oof_cb, pred_cb),
          'xgb': (oof_xgb, pred_xgb), 'mlp': (oof_mlp, pred_mlp)}

stack_X    = np.column_stack([logit(to_rank(o)) for o, _ in models.values()])
stack_X_te = np.column_stack([logit(to_rank(p)) for _, p in models.values()])
lr = LogisticRegression(C=10, max_iter=500).fit(stack_X, y)
for n, c in zip(models, lr.coef_[0]):
    flag = '  drop (negative)' if c < 0 else ''
    print(f'  {n:<8} coef {c:+.4f}{flag}')

oof_stack  = lr.predict_proba(stack_X)[:, 1]
pred_stack = lr.predict_proba(stack_X_te)[:, 1]
auc_stack  = roc_auc_score(y, oof_stack)

candidates = {**{n: (roc_auc_score(y, o), p) for n, (o, p) in models.items()},
              'stacker': (auc_stack, pred_stack)}
for n, (a, _) in candidates.items():
    print(f'  {n:<8} OOF AUC {a:.5f}')

best_name = max(candidates, key=lambda k: candidates[k][0])
best_auc, best_pred = candidates[best_name]
best_oof = {**{n: o for n, (o, _) in models.items()}, 'stacker': oof_stack}[best_name]
print(f'best: {best_name}  OOF AUC {best_auc:.5f}')

iso = IsotonicRegression(out_of_bounds='clip').fit(best_oof, y)
final_pred = iso.transform(best_pred)
print(f'after isotonic: pred mean {final_pred.mean():.4f}  (train pos rate {y.mean():.4f})')

In [ ]:
raw_t, raw_p = calibration_curve(y, best_oof,                n_bins=20)
iso_t, iso_p = calibration_curve(y, iso.transform(best_oof), n_bins=20)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='perfect')
ax.plot(raw_p, raw_t, 'o-', label='raw', color='steelblue')
ax.plot(iso_p, iso_t, 's-', label='isotonic', color='darkorange')
ax.set(xlabel='predicted', ylabel='empirical rate', title=f'Reliability — {best_name}')
ax.legend(); plt.tight_layout(); plt.show()

## Submission

In [ ]:
out = (pd.DataFrame({'id': test_ids_sorted, TARGET: final_pred})
         .merge(test[['id']], on='id', how='right'))
out.to_csv('submission.csv', index=False)
print('wrote submission.csv', out.shape, 'mean', round(out[TARGET].mean(), 4))